In [55]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import math

In [56]:
file_path = 'vn100_featured_data.csv'

df = pd.read_csv(file_path)

In [57]:
def _get_hose_tick_size(price):
    if price < 10000:
        return 10
    elif price < 50000:
        return 50
    return 100


def _round_down_to_tick(price, tick):
    return math.floor(price / tick) * tick


def _round_up_to_tick(price, tick):
    return math.ceil(price / tick) * tick


def _calc_ceiling_floor(ref_price):
    if pd.isna(ref_price) or ref_price <= 0:
        return np.nan, np.nan

    limit = 0.07
    tick = _get_hose_tick_size(ref_price)

    raw_ceiling = ref_price * (1 + limit)
    raw_floor = ref_price * (1 - limit)

    ceiling = _round_up_to_tick(raw_ceiling, tick)
    floor = _round_down_to_tick(raw_floor, tick)
    return ceiling, floor


def _classify_price_state(close_today, ref_price, floor_price, ceiling_price, eps=1e-9):
    if pd.isna(close_today) or pd.isna(ref_price):
        return "UNKNOWN"

    if not pd.isna(ceiling_price) and abs(close_today - ceiling_price) < eps:
        return "CEILING"
    if not pd.isna(floor_price) and abs(close_today - floor_price) < eps:
        return "FLOOR"
    if abs(close_today - ref_price) < eps:
        return "REFERENCE"
    if close_today > ref_price:
        return "UP"
    if close_today < ref_price:
        return "DOWN"
    return "REFERENCE"


def _state_to_score(state, pct_change):
    pct_change = 0 if pd.isna(pct_change) else pct_change

    if state == "CEILING":
        return 7.0
    if state == "FLOOR":
        return -7.0
    if state == "REFERENCE":
        return 0.0

    return max(-6.99, min(6.99, pct_change))


def _score_to_color(score):
    """
    Tô màu kiểu bảng điện:
    sàn = cyan, giảm = đỏ, tham chiếu = vàng, tăng = xanh, trần = tím
    """
    if pd.isna(score):
        return "#9E9E9E"

    if score <= -7:
        return "#00C8FF"   # sàn
    elif score < 0:
        # đỏ nhạt -> đỏ đậm
        if score < -5:
            return "#FF4D4F"
        elif score < -3:
            return "#FF6B6B"
        else:
            return "#FF9B9B"
    elif score == 0:
        return "#FFD84D"   # tham chiếu
    elif score < 7:
        # xanh nhạt -> xanh đậm
        if score < 1:
            return "#6CCB8E"
        elif score < 3:
            return "#4CAF72"
        else:
            return "#2E8B57"
    else:
        return "#C000FF"   # trần


def _wrap_label(text, width=14):
    words = str(text).split()
    lines = []
    current = ""

    for w in words:
        test = f"{current} {w}".strip()
        if len(test) <= width:
            current = test
        else:
            if current:
                lines.append(current)
            current = w

    if current:
        lines.append(current)

    return "<br>".join(lines)


def create_treemap(df, top_n=10):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    dates = sorted(df["date"].dropna().unique(), reverse=True)
    if len(dates) < 2:
        raise ValueError("Cần ít nhất 2 ngày giao dịch để vẽ treemap.")

    ngay_nay = dates[0]
    ngay_truoc = dates[1]

    df_nay = df[df["date"] == ngay_nay].copy()
    df_truoc = df[df["date"] == ngay_truoc].copy()

    df_merged = pd.merge(
        df_nay[["ticker", "icb_name2", "LS_GiaDongCua", "LS_GiaTriKhopLenh"]],
        df_truoc[["ticker", "LS_GiaDongCua"]],
        on="ticker",
        suffixes=("_nay", "_truoc")
    )

    df_merged = df_merged.rename(columns={
        "LS_GiaDongCua_nay": "close_today",
        "LS_GiaDongCua_truoc": "close_prev",
        "LS_GiaTriKhopLenh": "traded_value"
    })

    df_merged["pct_change"] = (
        (df_merged["close_today"] - df_merged["close_prev"])
        / df_merged["close_prev"] * 100
    )

    ceil_floor = df_merged["close_prev"].apply(_calc_ceiling_floor)
    df_merged["ceiling_price"] = ceil_floor.apply(lambda x: x[0])
    df_merged["floor_price"] = ceil_floor.apply(lambda x: x[1])

    df_merged["price_state"] = df_merged.apply(
        lambda row: _classify_price_state(
            row["close_today"],
            row["close_prev"],
            row["floor_price"],
            row["ceiling_price"]
        ),
        axis=1
    )

    df_merged["color_score_stock"] = df_merged.apply(
        lambda row: _state_to_score(row["price_state"], row["pct_change"]),
        axis=1
    )

    def weighted_avg_color(group):
        val = group["traded_value"]
        score = group["color_score_stock"]
        if val.sum() == 0:
            return 0
        return (val * score).sum() / val.sum()

    def weighted_avg_pct(group):
        val = group["traded_value"]
        pct = group["pct_change"]
        if val.sum() == 0:
            return 0
        return (val * pct).sum() / val.sum()

    ind_stats = (
        df_merged.groupby("icb_name2", as_index=False)
        .apply(lambda g: pd.Series({
            "Color_Score": weighted_avg_color(g),
            "Avg_Pct_Change": weighted_avg_pct(g),
            "Total_Val": g["traded_value"].sum()
        }))
        .reset_index(drop=True)
    )

    ind_stats["GiaTri_TyVND"] = ind_stats["Total_Val"] / 1_000_000_000
    ind_stats["Avg_Pct_Change"] = ind_stats["Avg_Pct_Change"].round(2)

    ind_stats = (
        ind_stats.sort_values("Total_Val", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    ind_stats["label_show"] = ind_stats["icb_name2"].apply(lambda x: _wrap_label(x, 14))
    ind_stats["display_text"] = (
        ind_stats["label_show"] + "<br>" +
        ind_stats["Avg_Pct_Change"].map(lambda x: f"{x:+.2f}%")
    )
    ind_stats["color_hex"] = ind_stats["Color_Score"].apply(_score_to_color)

    fig = go.Figure(go.Treemap(
        labels=ind_stats["label_show"],
        parents=[""] * len(ind_stats),
        values=ind_stats["GiaTri_TyVND"],
        text=ind_stats["display_text"],
        textinfo="text",
        textfont=dict(
            size=22,
            color="white",
            family="Arial"
        ),
        textposition="middle center",
        marker=dict(
            colors=ind_stats["color_hex"],
            line=dict(color="white", width=2)
        ),
        branchvalues="total",
        root=dict(color="white"),
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Biến động: %{customdata[0]:+.2f}%<br>"
            "GTGD: %{customdata[1]:,.0f} tỷ"
            "<extra></extra>"
        ),
        customdata=np.column_stack([
            ind_stats["Avg_Pct_Change"],
            ind_stats["GiaTri_TyVND"]
        ]),
        tiling=dict(
            pad=2,
            packing="squarify"
        ),
        sort=True
    ))

    fig.update_layout(
        title=dict(
            text=f"Biểu đồ ngành",
            x=0.5,
            xanchor='center',
            y=0.98,
            yanchor='top',
            font=dict(size=28, color='#1F1F1F', family='Arial')
        ),
        template="plotly_white",
        margin=dict(t=10, l=10, r=10, b=10),
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=1280,
        height=900
    )

    return fig

# ===== Refactor block: thay từ prepare(...) đến create_top_losers_table(...) =====
TABLE_WHITE_STYLE = dict(fill_color='white', align='left', line=dict(width=0))
TABLE_LAYOUT = dict(
    width=500,
    height=400,
    margin=dict(t=40, l=10, r=10, b=10),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
RANK_TABLE_LAYOUT = dict(
    height=500,
    width=800,
    margin=dict(t=100, l=10, r=10, b=10),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
NET_CHART_LAYOUT = dict(
    height=550,
    width=1000,
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    margin=dict(t=100, l=50, r=50, b=50)
)


def _get_latest_dates(df, reverse=True):
    return sorted(df['date'].unique(), reverse=reverse)


def _split_by_date(df, date_now, date_prev=None):
    df_now = df[df['date'] == date_now].copy()
    if date_prev is None:
        return df_now, None
    df_prev = df[df['date'] == date_prev].copy()
    return df_now, df_prev


def _build_latest_change_df(df):
    dates = _get_latest_dates(df, reverse=True)
    ngay_nay, ngay_truoc = dates[:2]
    df_nay, df_truoc = _split_by_date(df, ngay_nay, ngay_truoc)

    df_merge = pd.merge(
        df_nay[['ticker', 'LS_GiaDongCua', 'LS_KhoiLuongKhopLenh', 'LS_GiaTriKhopLenh']],
        df_truoc[['ticker', 'LS_GiaDongCua']],
        on='ticker',
        suffixes=('_nay', '_truoc')
    )
    df_merge['pct_change'] = (
        (df_merge['LS_GiaDongCua_nay'] - df_merge['LS_GiaDongCua_truoc'])
        / df_merge['LS_GiaDongCua_truoc'] * 100
    )
    df_merge['price_change'] = df_merge['LS_GiaDongCua_nay'] - df_merge['LS_GiaDongCua_truoc']
    return df_merge, ngay_nay


def prepare(df_input, is_value=False):
    df_out = df_input.copy()
    df_out['Giá'] = df_out['LS_GiaDongCua_nay']
    df_out['%'] = df_out['pct_change'].round(2)
    df_out['+/-'] = df_out['price_change'].round(2)

    metric_col = 'Giá trị (Tỷ)' if is_value else 'Khối lượng'
    metric_src = 'LS_GiaTriKhopLenh' if is_value else 'LS_KhoiLuongKhopLenh'
    df_out[metric_col] = df_out[metric_src] / 1e9 if is_value else df_out[metric_src]
    return df_out[['ticker', 'Giá', '%', '+/-', metric_col]]


def create_table(df_display, col_names):
    return go.Table(
        header=dict(values=col_names, **TABLE_WHITE_STYLE, font=dict(size=13)),
        cells=dict(
            values=[df_display[col] for col in df_display.columns],
            **TABLE_WHITE_STYLE,
            font=dict(size=12)
        )
    )


def _make_table_figure(df_display, headers, title, layout_kwargs=None):
    fig = go.Figure(data=[go.Table(
        header=dict(values=headers, **TABLE_WHITE_STYLE),
        cells=dict(values=[df_display[col] for col in df_display.columns], **TABLE_WHITE_STYLE)
    )])
    fig.update_layout(title=title, **(layout_kwargs or TABLE_LAYOUT))
    return fig


def _create_top_table(df, *, sort_col, is_value, title_fmt):
    df_merge, ngay_nay = _build_latest_change_df(df)
    top_df = df_merge.sort_values(by=sort_col, ascending=False).head(10)
    df_display = prepare(top_df, is_value=is_value)
    headers = ['Mã', 'Giá', '%', '+/-', 'Giá trị (Tỷ)' if is_value else 'Khối lượng']
    return _make_table_figure(df_display, headers, title_fmt.format(ngay_nay=ngay_nay))


def create_top_volume_table(df):
    return _create_top_table(
        df,
        sort_col='LS_KhoiLuongKhopLenh',
        is_value=False,
        title_fmt='Top 10 Khối lượng giao dịch({ngay_nay})'
    )


def create_top_value_table(df):
    return _create_top_table(
        df,
        sort_col='LS_GiaTriKhopLenh',
        is_value=True,
        title_fmt='Top 10 Giá trị giao dịch ({ngay_nay})'
    )


def _latest_net_values(df, net_col, compute_net=None):
    ngay_nay = _get_latest_dates(df, reverse=True)[0]
    df_nay, _ = _split_by_date(df, ngay_nay)
    if compute_net is not None:
        df_nay[net_col] = compute_net(df_nay)

    top_buy = df_nay[df_nay[net_col] > 0].sort_values(by=net_col, ascending=False).head(10)
    top_sell = df_nay[df_nay[net_col] < 0].sort_values(by=net_col, ascending=True).head(10)

    buy_vals = (top_buy[net_col] / 1_000_000_000).tolist()
    sell_vals = (top_sell[net_col].abs() / 1_000_000_000).tolist()

    return {
        'ngay_nay': ngay_nay,
        'buy_tickers': top_buy['ticker'].tolist(),
        'buy_vals': buy_vals,
        'sell_tickers': top_sell['ticker'].tolist(),
        'sell_vals': sell_vals,
        'max_val': max(max(buy_vals) if buy_vals else 0, max(sell_vals) if sell_vals else 0)
    }


def _create_net_chart(df, *, net_col, title_html, subplot_titles, compute_net=None):
    data = _latest_net_values(df, net_col=net_col, compute_net=compute_net)
    fig = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=False,
        horizontal_spacing=0.12,
        subplot_titles=subplot_titles
    )

    for col, x_vals, y_vals, name, color in [
        (1, data['sell_vals'], data['sell_tickers'], 'Bán ròng', '#ff6666'),
        (2, data['buy_vals'], data['buy_tickers'], 'Mua ròng', '#00cc66')
    ]:
        fig.add_trace(
            go.Bar(
                x=x_vals,
                y=y_vals,
                orientation='h',
                marker=dict(color=color),
                text=[f'{v:.1f}T' for v in x_vals],
                textposition='outside',
                name=name,
                cliponaxis=False
            ),
            row=1,
            col=col
        )

    axis_max = data['max_val'] * 1.3
    fig.update_xaxes(range=[axis_max, 0], visible=False, row=1, col=1)
    fig.update_xaxes(range=[0, axis_max], visible=False, row=1, col=2)
    fig.update_yaxes(autorange='reversed', side='right', tickfont=dict(size=12, family='Arial white'), row=1, col=1)
    fig.update_yaxes(autorange='reversed', side='left', tickfont=dict(size=12, family='Arial white'), row=1, col=2)
    fig.update_layout(
        title=dict(
            text=title_html.format(ngay_nay=data['ngay_nay']),
            x=0.5,
            font=dict(size=18, color='black')
        ),
        **NET_CHART_LAYOUT
    )
    return fig


def create_foreign_net_chart(df):
    return _create_net_chart(
        df,
        net_col='NN_net',
        compute_net=lambda x: x['KN_GTDGRong'],
        title_html='<b>TOP 10 MUA/BÁN RÒNG KHỐI NGOẠI (VN100)</b><br>Ngày: {ngay_nay}',
        subplot_titles=('Bán ròng', 'Mua ròng')
    )


def create_self_net_chart(df):
    return _create_net_chart(
        df,
        net_col='TD_net',
        compute_net=lambda x: x['TD_GtMua'] - x['TD_GtBan'],
        title_html='<b>TOP 10 MUA/BÁN RÒNG TỰ DOANH (VN100)</b><br>Ngày: {ngay_nay}',
        subplot_titles=('Tự doanh Bán ròng', 'Tự doanh Mua ròng')
    )


def _calc_rank_table(df, n_days, dates, *, ascending):
    if len(dates) <= n_days:
        return None

    day_now = dates[-1]
    day_prev = dates[-(n_days + 1)]
    df_now, df_prev = _split_by_date(df, day_now, day_prev)

    df_merge = pd.merge(
        df_now[['ticker', 'LS_GiaDongCua']],
        df_prev[['ticker', 'LS_GiaDongCua']],
        on='ticker',
        suffixes=('_now', '_prev')
    )
    df_merge['pct'] = (
        (df_merge['LS_GiaDongCua_now'] - df_merge['LS_GiaDongCua_prev'])
        / df_merge['LS_GiaDongCua_prev'] * 100
    )
    df_merge['change'] = df_merge['LS_GiaDongCua_now'] - df_merge['LS_GiaDongCua_prev']

    df_vol = (
        df[(df['date'] >= day_prev) & (df['date'] <= day_now)]
        .groupby('ticker')['LS_KhoiLuongKhopLenh']
        .mean()
        .reset_index()
    )
    df_merge = pd.merge(df_merge, df_vol, on='ticker', how='left')
    return df_merge.sort_values(by='pct', ascending=ascending).head(10)


def calc_top_table(df, n_days, dates):
    return _calc_rank_table(df, n_days, dates, ascending=False)


def calc_bottom_table(df, n_days, dates):
    return _calc_rank_table(df, n_days, dates, ascending=True)


def _format_rank_table(data, *, signed):
    pct_fmt = (lambda x: f'{x:+.2f}%') if signed else (lambda x: f'{x:.2f}%')
    chg_fmt = (lambda x: f'{x:+.2f}') if signed else (lambda x: f'{x:.2f}')
    return [
        data['ticker'],
        data['LS_GiaDongCua_now'].round(2),
        data['pct'].map(pct_fmt),
        data['change'].map(chg_fmt),
        data['LS_KhoiLuongKhopLenh'].map(lambda x: f'{x:,.0f}')
    ]


def format_table_top(data):
    return _format_rank_table(data, signed=True)


def format_table_bottom(data):
    return _format_rank_table(data, signed=False)


def _build_rank_dropdown(title, periods, datasets, formatter, dropdown_titles):
    fig = go.Figure()
    headers = ['Mã', 'Giá', '%', '+/-', 'KL TB']

    for i, data in enumerate(datasets):
        fig.add_trace(go.Table(
            header=dict(values=headers, **TABLE_WHITE_STYLE),
            cells=dict(values=formatter(data), **TABLE_WHITE_STYLE),
            visible=(i == 0)
        ))

    fig.update_layout(
        updatemenus=[dict(
            buttons=[
                dict(
                    label=label,
                    method='update',
                    args=[
                        {'visible': [i == idx for i in range(len(datasets))]},
                        {'title': dropdown_title}
                    ]
                )
                for idx, (label, dropdown_title) in enumerate(zip(periods, dropdown_titles))
            ],
            direction='down',
            x=0.98,
            xanchor='right',
            y=1.18,
            yanchor='top'
        )],
        title=dict(text=title, x=0.5, xanchor='center'),
        **RANK_TABLE_LAYOUT
    )
    return fig


def create_top_gainers_table(df):
    dates = _get_latest_dates(df, reverse=False)
    datasets = [
        calc_top_table(df, 1, dates),
        calc_top_table(df, 7, dates),
        calc_top_table(df, 30, dates),
    ]
    return _build_rank_dropdown(
        title='Top tăng giá',
        periods=['1 Ngày', '1 Tuần', '1 Tháng'],
        datasets=datasets,
        formatter=format_table_top,
        dropdown_titles=['Top tăng giá - 1 Ngày', 'Top tăng giá - 1 Tuần', 'Top tăng giá - 1 Tháng']
    )


def create_top_losers_table(df):
    dates = _get_latest_dates(df, reverse=False)
    datasets = [
        calc_bottom_table(df, 1, dates),
        calc_bottom_table(df, 5, dates),
        calc_bottom_table(df, 20, dates),
    ]
    return _build_rank_dropdown(
        title='Top giảm giá',
        periods=['1 Ngày', '1 Tuần', '1 Tháng'],
        datasets=datasets,
        formatter=format_table_bottom,
        dropdown_titles=['Top giảm giá - 1 Ngày', 'Top giảm giá - 1 Tuần', 'Top giảm giá - 1 Tháng']
    )
def _pick_column(df, explicit_name, candidates, label):
    if explicit_name is not None:
        if explicit_name not in df.columns:
            raise ValueError(f"Không tìm thấy cột {label}: {explicit_name}")
        return explicit_name

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(f"Không tìm thấy cột {label}. Hãy truyền tên cột thủ công.")


def _prepare_money_flow_features(
    df,
    close_col='LS_GiaDongCua',
    volume_col='LS_KhoiLuongKhopLenh',
    high_col=None,
    low_col=None,
):
    data = df.copy()
    data['date'] = pd.to_datetime(data['date'])

    high_col = _pick_column(
        data,
        high_col,
        ['LS_GiaCaoNhat', 'high', 'High', 'HIGH'],
        'high'
    )
    low_col = _pick_column(
        data,
        low_col,
        ['LS_GiaThapNhat', 'low', 'Low', 'LOW'],
        'low'
    )

    required = ['ticker', 'date', close_col, volume_col, high_col, low_col]
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f"Thiếu cột: {missing}")

    data = data.sort_values(['ticker', 'date']).reset_index(drop=True)
    g = data.groupby('ticker', sort=False)

    data['return_5d'] = g[close_col].pct_change(5) * 100

    data['traded_value'] = data[close_col] * data[volume_col]
    data['avg_value_20'] = (
        g['traded_value']
        .rolling(20, min_periods=20)
        .mean()
        .reset_index(level=0, drop=True)
    )

    data['value_ratio_20'] = data['traded_value'] / data['avg_value_20']
    data['value_ratio_20'] = data['value_ratio_20'].replace([np.inf, -np.inf], np.nan)

    data['typical_price'] = (data[high_col] + data[low_col] + data[close_col]) / 3
    data['raw_money_flow'] = data['typical_price'] * data[volume_col]

    tp_prev = g['typical_price'].shift(1)

    data['positive_mf'] = np.where(
        data['typical_price'] > tp_prev,
        data['raw_money_flow'],
        0.0
    )
    data['negative_mf'] = np.where(
        data['typical_price'] < tp_prev,
        data['raw_money_flow'],
        0.0
    )

    data['pos_mf_14'] = (
        g['positive_mf']
        .rolling(14, min_periods=14)
        .sum()
        .reset_index(level=0, drop=True)
    )
    data['neg_mf_14'] = (
        g['negative_mf']
        .rolling(14, min_periods=14)
        .sum()
        .reset_index(level=0, drop=True)
    )

    money_flow_ratio = data['pos_mf_14'] / data['neg_mf_14'].replace(0, np.nan)
    data['mfi14'] = 100 - (100 / (1 + money_flow_ratio))

    data.loc[(data['neg_mf_14'] == 0) & (data['pos_mf_14'] > 0), 'mfi14'] = 100
    data.loc[(data['pos_mf_14'] == 0) & (data['neg_mf_14'] > 0), 'mfi14'] = 0
    data.loc[(data['pos_mf_14'] == 0) & (data['neg_mf_14'] == 0), 'mfi14'] = 50
    data['mfi14'] = data['mfi14'].clip(0, 100)

    spread = (data[high_col] - data[low_col]).replace(0, np.nan)

    data['money_flow_multiplier'] = (
        ((data[close_col] - data[low_col]) - (data[high_col] - data[close_col])) / spread
    )
    data['money_flow_multiplier'] = (
        data['money_flow_multiplier']
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(-1, 1)
    )

    data['money_flow_volume'] = data['money_flow_multiplier'] * data[volume_col]

    mfv_21 = (
        g['money_flow_volume']
        .rolling(21, min_periods=21)
        .sum()
        .reset_index(level=0, drop=True)
    )
    vol_21 = (
        g[volume_col]
        .rolling(21, min_periods=21)
        .sum()
        .reset_index(level=0, drop=True)
    )

    data['cmf21'] = (mfv_21 / vol_21).replace([np.inf, -np.inf], np.nan)

    latest = g.tail(1).copy()
    latest = latest.dropna(subset=['return_5d', 'avg_value_20', 'mfi14', 'cmf21'])

    latest['ret_norm'] = latest['return_5d'].clip(-10, 10) / 10
    latest['mfi_norm'] = (latest['mfi14'] - 50) / 50
    latest['cmf_norm'] = latest['cmf21'].clip(-1, 1)

    latest['flow_score'] = 100 * (
        0.50 * latest['cmf_norm'] +
        0.30 * latest['mfi_norm'] +
        0.20 * latest['ret_norm']
    )

    latest = latest.sort_values('flow_score', ascending=False).reset_index(drop=True)

    return latest


def plot_top_flow_score(
    df,
    top_n=10,
    close_col='LS_GiaDongCua',
    volume_col='LS_KhoiLuongKhopLenh',
    high_col=None,
    low_col=None,
):
    features = _prepare_money_flow_features(
        df=df,
        close_col=close_col,
        volume_col=volume_col,
        high_col=high_col,
        low_col=low_col,
    )

    if features.empty:
        print("Không có dữ liệu để vẽ biểu đồ dòng tiền.")
        return

    top_df = features.head(top_n).sort_values('flow_score', ascending=True)

    fig = px.bar(
        top_df,
        x='flow_score',
        y='ticker',
        orientation='h',
        title='Top cổ phiếu có dòng tiền mạnh'
    )

    fig.update_layout(
        xaxis_title='Điểm dòng tiền',
        yaxis_title='Mã cổ phiếu'
    )

    fig.show()


In [58]:
fig = create_treemap(df)
fig.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_14376\799020558.py:181: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [59]:
fig = create_top_volume_table(df)
fig.show()

In [60]:
fig = create_top_value_table(df)
fig.show()

In [61]:
fig = create_foreign_net_chart(df)
fig.show()

In [62]:
fig = create_self_net_chart(df)
fig.show()

In [63]:
fig = create_top_gainers_table(df)
fig.show()

In [64]:
fig = create_top_losers_table(df)
fig.show()

In [65]:
plot_top_flow_score(df)